# 🎙️ Qwen3-TTS Professional Clone Studio — T4 Sampling Fix v3

**For:** Google Colab **Tesla T4 16GB VRAM** • `Qwen/Qwen3-TTS-12Hz-0.6B-Base` • **FP32 + SDPA**, default CUDA graphs.

### Why v3?

Version 2 successfully loads weights and captures CUDA graphs but some FP16 Base generations trigger CUDA invalid-probability assertions (`nan`, `inf`). This version **uses FP32 for numerical stability**; it does **not promise** the FP16 T4 benchmark speed. If the FP32 graphs use too much VRAM, select the **official FP32 (no graphs)** fallback in the launch cell below.

**Important:** After any CUDA device-side assert, use **Runtime → Restart session** before retrying. Run all cells in order on the newly started session. Cached Drive voices, MP3s, and old checkpoints are not deleted. This version uses an independent new cache namespace for FP32 generation.

### Features

Single/batch TXT, accurate cloning with reference transcripts (ICL/x-vector), Whisper auto-transcription, portable voice ZIP import/export, live 5-second voice refresh, sentence-aware chunks, audio validation, retries, resume checkpoints, MP3 mastering, multilingual generation, one GPU worker. **Actual T4 audio generation has not been tested by the notebook author.**


In [ ]:
# STEP 1 — GPU / driver diagnostics (must show Tesla T4)
import os, sys, subprocess
print("Python:",sys.version)
subprocess.run(["nvidia-smi"], check=True)
from importlib.metadata import version, PackageNotFoundError
try:
    print("Preinstalled PyTorch:", version("torch"))
except PackageNotFoundError:
    print("PyTorch is not installed yet")
print("CUDA runtime checks run in STEP 3 after package installation")


In [ ]:
# STEP 2 — Install pinned, Qwen-compatible dependencies + required audio tools
# Fix for MimiConfig.rope_theta: transformers 5.17 is incompatible with this
# version of the Qwen TTS Mimi codec; use the tested 5.15.1 release.
# SoX is also needed for some audio-decode paths in Qwen's dependencies.
import os, sys, subprocess, shutil
if shutil.which('ffmpeg') is None or shutil.which('sox') is None:
    subprocess.run(['apt-get','-qq','update'], check=True)
    subprocess.run(['apt-get','-qq','install','-y','ffmpeg','sox','libsox-fmt-all'], check=True)
packages = [
    'faster-qwen3-tts',
    'qwen-tts-hf==0.1.1.post1',
    'transformers==5.15.1', # Pinned: prevents MimiConfig.rope_theta crash
    'gradio>=5.0,<7', 'soundfile>=0.12',
    'openai-whisper', 'numpy', 'huggingface-hub',
]
subprocess.run([sys.executable, '-m','pip','install','-q','--upgrade',*packages],check=True)
from importlib.metadata import version
print('transformers:', version('transformers'))
print('qwen-tts-hf:', version('qwen-tts-hf'))
print('faster-qwen3-tts:',version('faster-qwen3-tts'))
assert version('transformers') == '5.15.1', 'The required transformers version was not installed.'
assert version('qwen-tts-hf') == '0.1.1.post1', 'The required qwen-tts-hf version was not installed.'
print('Audio tools:',shutil.which('sox'),shutil.which('ffmpeg'))
print('✅ Compatible dependencies installed. If you upgraded an existing runtime,')
print('   use Runtime → Restart session before running STEP 3 onward.')


### Optional CUDA driver repair — **skip unless CUDA initialization fails**

Some T4 hosts expose only CUDA **12.4** drivers, but PyPI may install a PyTorch wheel targeting a newer driver. The [`faster-qwen3-tts` README](https://github.com/andimarafioti/faster-qwen3-tts#install) documents using PyTorch **2.5.1 cu124** when the driver is limited to CUDA 12.4. Only run the following repair cell in that specific situation. Restart the runtime after changing PyTorch, then repeat the notebook. For a CUDA driver older than 12.4, choose a matching wheel supported by your Python version instead.


In [ ]:
# OPTIONAL REPAIR CELL — only for CUDA-12.4 driver hosts with a newer incompatible PyTorch CUDA wheel
# %pip install -q --force-reinstall torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
# After uncommenting and running, Runtime → Restart session, then rerun from STEP 1.
print('Skipped. Uncomment only if the 12.4-driver mismatch is confirmed.')


In [ ]:
# STEP 3 — Mount Drive and choose persistent storage folder
import os, pathlib
from google.colab import drive
drive.mount('/content/drive',force_remount=False)
os.environ['QWEN_STUDIO_BASE']='/content/drive/MyDrive/Qwen3_Production'
os.environ['QWEN_STUDIO_TEMP']='/content/qwen3_studio_temp'
pathlib.Path(os.environ['QWEN_STUDIO_BASE']).mkdir(parents=True,exist_ok=True)
import torch
assert torch.cuda.is_available(), 'CUDA GPU unavailable. Check STEP 1 and the optional 12.4-driver repair cell.'
print('Device:',torch.cuda.get_device_name(0))
print('Studio Drive:',os.environ['QWEN_STUDIO_BASE'])


In [ ]:
%%writefile /content/qwen3_t4_studio_app.py
import argparse
import concurrent.futures
import contextlib
import gc
import hashlib
import importlib.util
import json
import os
import random
import re
import shutil
import subprocess
import sys
import tempfile
import threading
import time
import uuid
import zipfile
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
try:
    import gradio as gr
except ImportError:
    gr = None

MODEL_REPO = "Qwen/Qwen3-TTS-12Hz-0.6B-Base"
BASE = Path(os.environ.get("QWEN_STUDIO_BASE", "/content/drive/MyDrive/Qwen3_Production"))
LOCAL_TEMP = Path(os.environ.get("QWEN_STUDIO_TEMP", "/content/qwen3_studio_temp"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
VOICE_DIR, OUTPUT_DIR, CACHE_DIR, EXPORT_DIR = [BASE / x for x in ("voices", "outputs", "batch_cache", "voice_exports")]
for p in (BASE, VOICE_DIR, OUTPUT_DIR, CACHE_DIR, EXPORT_DIR, LOCAL_TEMP):
    p.mkdir(parents=True, exist_ok=True)
MODEL_LOCK = threading.RLock()
VOICE_IO_LOCK = threading.RLock()
MODEL = None
SAMPLE_RATE = 24000
GPU_NAME, GPU_VRAM_GB = "CPU", 0.0
MODEL_SNAPSHOT_ID = "qwen3-0.6b-unknown"
ACTIVE_VOICE = None
MODEL_NORMALIZATION_AVAILABLE = True
VOICE_FORMAT = "qwen3_clone_studio_v1"
MAX_IMPORT_FILES, MAX_IMPORT_BYTES = 60, 512 * 1024 * 1024
MP3_BITRATE, DEFAULT_PAUSE_MS = "192k", 90
CACHE_VERSION = "qwen3_t4_fp32_sampling_v3"
MODEL_DTYPE = torch.float32  # T4 has no native BF16; Qwen Base can produce non-finite FP16 probabilities.
INFERENCE_BACKEND = os.getenv("QWEN_T4_BACKEND", "graphs").strip().lower()
if INFERENCE_BACKEND not in ("graphs", "official"):
    raise ValueError("QWEN_T4_BACKEND must be graphs or official.")
QUALITY_GUARD_VERSION = "audio_guard_v2"
QUALITY_PRESETS = {
    "Fast (1 attempt)": {"retries": 0},
    "Reliable (2 attempts)": {"retries": 1},
    "Extra careful (3 attempts)": {"retries": 2},
}
DEFAULT_QUALITY = "Reliable (2 attempts)"
LANGUAGES = ("Voice default", "English", "Spanish", "French", "German", "Portuguese", "Italian", "Russian", "Chinese", "Japanese", "Korean")
LANGUAGE_NAMES = {
    "en":"English", "english":"English", "es":"Spanish", "spanish":"Spanish",
    "fr":"French", "french":"French", "de":"German", "german":"German",
    "zh":"Chinese", "zh-cn":"Chinese", "chinese":"Chinese",
    "ja":"Japanese", "japanese":"Japanese", "ko":"Korean", "korean":"Korean",
    "pt":"Portuguese", "portuguese":"Portuguese", "it":"Italian", "italian":"Italian",
    "ru":"Russian", "russian":"Russian",
}
VOICE_MODES = ("High fidelity (ICL)", "Fast multilingual (x-vector)")
DEFAULT_VOICE_MODE = VOICE_MODES[0]

def _validate_mode(mode):
    mode = mode or DEFAULT_VOICE_MODE
    if mode not in VOICE_MODES:
        raise ValueError(f"Unknown cloning mode: {mode}")
    return mode

def language_name(value):
    name = LANGUAGE_NAMES.get(str(value or "en").strip().lower())
    if name is None:
        raise ValueError("Qwen3-TTS supports 10 languages; select one from the language dropdown.")
    return name

def resolve_language(meta, override):
    return language_name(meta.get("language") if not override or override == "Voice default" else override)


def _safe_name(value, max_len=100):
    value = str(value or "").strip()
    value = re.sub(r'[\\/:*?"<>|]+', "_", value)
    value = re.sub(r"\s+", " ", value).strip(" .")
    if not value:
        raise ValueError("Name cannot be empty.")
    return value[:max_len]


def _safe_stem(name):
    stem = Path(str(name)).stem.strip()
    stem = re.sub(r"[^A-Za-z0-9._ -]+", "_", stem)
    stem = re.sub(r"\s+", " ", stem).strip(" .")
    return (stem or "voiceover")[:100]


def _coerce_path(value):
    if value is None:
        return None
    if isinstance(value, (str, os.PathLike)):
        return Path(value)
    for attr in ("path", "name"):
        candidate = getattr(value, attr, None)
        if candidate:
            return Path(candidate)
    raise TypeError(f"Unsupported uploaded-file object: {type(value).__name__}")


def _clean_text(text):
    text = str(text or "").replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def clean_script(text):
    text = str(text or "").replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r" *\n+ *", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def _sha256_file(path, block=1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            data = f.read(block)
            if not data:
                break
            h.update(data)
    return h.hexdigest()


def _atomic_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        tmp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
        os.replace(tmp, path)
    finally:
        tmp.unlink(missing_ok=True)


def _atomic_copy(src, dst):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(f".{dst.name}.{uuid.uuid4().hex}.tmp")
    try:
        shutil.copy2(src, tmp)
        if tmp.stat().st_size <= 0:
            raise RuntimeError(f"Atomic copy produced an empty file: {dst.name}")
        os.replace(tmp, dst)
    finally:
        tmp.unlink(missing_ok=True)




def _release_memory(force=False):
    gc.collect()
    if not torch.cuda.is_available():
        return
    should_clear = bool(force)
    if not should_clear:
        try:
            free_b, total_b = torch.cuda.mem_get_info()
            should_clear = free_b < max(1024**3, int(total_b * 0.08))
        except Exception:
            should_clear = False
    if should_clear:
        _REAL_EMPTY_CACHE()


def _is_fatal_cuda_error(exc):
    msg = str(exc).lower()
    return any(s in msg for s in (
        "device-side assert", "illegal memory access", "cuda error: an illegal",
        "unspecified launch failure", "misaligned address", "probability tensor contains",
    ))





def _resolve_snapshot_id():
    """Cache model revision on Drive so an offline restart can still resume jobs."""
    revision_file = BASE / "qwen3_model_revision.txt"
    try:
        from huggingface_hub import model_info
        rev = str(model_info(MODEL_REPO).sha)[:16]
        if rev:
            revision_file.write_text(rev, encoding="utf-8")
            return rev
    except Exception as exc:
        print(f"Model revision lookup offline: {exc}", flush=True)
    if revision_file.is_file():
        old = revision_file.read_text(encoding="utf-8").strip()
        if old:
            return old
    return hashlib.sha256(MODEL_REPO.encode()).hexdigest()[:16]


class _OfficialFP32Adapter:
    """Use the original Qwen API if GPU memory won't accommodate FP32 graphs.

    Set QWEN_T4_BACKEND=official BEFORE launching the studio process.
    Do not attempt to recover an existing CUDA device-side assertion in-process.
    """
    sample_rate = 24000

    def __init__(self, qwen_model):
        self.model = qwen_model
        self._voice_prompt_cache = {}

    def warmup(self, prefill_len=120):
        pass  # Official backend does not capture CUDA graphs.

    def generate_voice_clone(self, *, text, language, ref_audio, ref_text,
                             xvec_only, append_silence, max_new_tokens,
                             temperature, top_k, top_p, repetition_penalty,
                             do_sample):
        # Official Qwen API calls this x_vector_only_mode, whereas the Faster
        # backend calls it xvec_only. Do NOT forward append_silence, which is
        # not supported as a generation flag in the official implementation.
        return self.model.generate_voice_clone(
            text=text, language=language, ref_audio=ref_audio,
            ref_text=ref_text, x_vector_only_mode=xvec_only,
            non_streaming_mode=False, max_new_tokens=max_new_tokens,
            temperature=temperature, top_k=top_k, top_p=top_p,
            repetition_penalty=repetition_penalty, do_sample=do_sample,
        )


def _is_out_of_memory(exc):
    return "out of memory" in str(exc).lower() or isinstance(exc, torch.cuda.OutOfMemoryError)


def _cuda_launch_guidance(exc):
    if _is_out_of_memory(exc):
        return ("Insufficient VRAM for the FP32 CUDA-graph backend. Restart the Colab "
                "session, then in the launch cell set QWEN_T4_BACKEND=official to "
                "use the non-graph FP32 backend. Your Drive voices are preserved.")
    if _is_fatal_cuda_error(exc):
        return ("The CUDA context may be corrupted (device-side assertion). "
                "Restart the Colab session before rerunning. This notebook "
                "uses FP32 to avoid the known FP16 invalid-probability problem.")
    return str(exc)


def load_model():
    global MODEL, SAMPLE_RATE, GPU_NAME, GPU_VRAM_GB, MODEL_SNAPSHOT_ID
    if MODEL is not None:
        return MODEL
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available. Set Colab Runtime > Change runtime type > T4 GPU. "
                           "If a driver mismatch is shown after installing packages, run the CUDA repair cell.")
    GPU_NAME = torch.cuda.get_device_name(0)
    GPU_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if GPU_VRAM_GB < 12:
        raise RuntimeError(f"{GPU_NAME} has only {GPU_VRAM_GB:.1f} GiB VRAM; 12+ GiB is recommended.")
    # Fail before downloading weights when transformers is incompatible with the
    # Qwen-TTS Mimi codec. Transformers 5.17 raises MimiConfig.rope_theta.
    # Pin from faster-qwen3-tts upstream's working environment.
    from importlib.metadata import version, PackageNotFoundError
    try:
        tf_version = version("transformers")
        qwen_dist_version = version("qwen-tts-hf")
    except PackageNotFoundError as exc:
        raise RuntimeError(
            "Incomplete Qwen setup. Rerun the notebook installation cell: "
            "pip install 'faster-qwen3-tts' 'qwen-tts-hf==0.1.1.post1' "
            "'transformers==5.15.1'"
        ) from exc
    print(f"Qwen compatibility: transformers={tf_version}, qwen-tts-hf={qwen_dist_version}", flush=True)
    if tf_version != "5.15.1":
        raise RuntimeError(
            f"Incompatible transformers={tf_version}: Qwen's Mimi tokenizer can crash "
            "with AttributeError: MimiConfig.rope_theta. Run the patched "
            "install cell (transformers==5.15.1), then restart Colab's "
            "runtime before launching the studio."
        )
    if qwen_dist_version != "0.1.1.post1":
        raise RuntimeError(
            f"Unexpected qwen-tts-hf={qwen_dist_version}; install "
            "qwen-tts-hf==0.1.1.post1, then restart Colab's runtime."
        )
    # Tesla T4 has FP16 and FP32, but not native BF16. In the Qwen Base clone
    # model FP16 can overflow sampling logits and poison CUDA multinomial
    # probabilities. FP32 avoids that numerical risk (at a speed/VRAM cost).
    MODEL_SNAPSHOT_ID = _resolve_snapshot_id()
    print(f"Loading {MODEL_REPO} | {GPU_NAME} {GPU_VRAM_GB:.1f} GiB "
          f"| FP32 / SDPA | backend={INFERENCE_BACKEND}", flush=True)
    with MODEL_LOCK:
        try:
            if INFERENCE_BACKEND == "graphs":
                from faster_qwen3_tts import FasterQwen3TTS
                candidate = FasterQwen3TTS.from_pretrained(
                    MODEL_REPO, device="cuda", dtype=MODEL_DTYPE,
                    attn_implementation="sdpa", backend="torch", max_seq_len=768,
                )
                # Capture in FP32 to prevent FP16 decoding overflow. This
                # consumes more GPU memory than the old FP16 version.
                candidate.warmup(prefill_len=96)
            else:
                from qwen_tts import Qwen3TTSModel
                official = Qwen3TTSModel.from_pretrained(
                    MODEL_REPO, device_map="cuda:0", torch_dtype=MODEL_DTYPE,
                    attn_implementation="sdpa",
                )
                candidate = _OfficialFP32Adapter(official)
            sample_rate = int(candidate.sample_rate)
            if not (16000 <= sample_rate <= 48000):
                raise RuntimeError(f"Unexpected Qwen3-TTS sample rate: {sample_rate}")
            MODEL, SAMPLE_RATE = candidate, sample_rate
        except Exception as exc:
            # Never silently try new CUDA operations after an asynchronous
            # device-side assert; the CUDA context must be recreated instead.
            raise RuntimeError(_cuda_launch_guidance(exc)) from exc
    print(f"✅ Qwen3-TTS ready | {SAMPLE_RATE} Hz | FP32 "
          f"| {INFERENCE_BACKEND} | revision {MODEL_SNAPSHOT_ID}", flush=True)
    return MODEL


# Optional CPU Whisper auto-transcript (never loaded into T4 GPU).

def transcribe_reference_audio(ref_audio, language="en"):
    src = _coerce_path(ref_audio)
    if not src or not src.exists():
        raise FileNotFoundError("Upload the reference recording first.")
    import whisper
    lang = str(language or "").strip().lower()
    aliases = {"english":"en", "spanish":"es", "french":"fr", "german":"de",
        "chinese":"zh", "japanese":"ja", "korean":"ko", "portuguese":"pt", "italian":"it", "russian":"ru"}
    iso = aliases.get(lang, lang)
    if iso not in {"en", "es", "fr", "de", "zh", "ja", "ko", "pt", "it", "ru"}:
        iso = None
    asr = whisper.load_model("base", device="cpu")
    try:
        result = asr.transcribe(str(src), language=iso, fp16=False, condition_on_previous_text=False)
        transcript = _clean_text(result.get("text", ""))
        if not transcript:
            raise RuntimeError("Whisper returned an empty transcript.")
        return transcript
    finally:
        del asr
        gc.collect()


def _audio_duration(path):
    info = sf.info(str(path))
    return float(info.frames) / float(info.samplerate)


def _convert_reference_audio(src, dst):
    src, dst = Path(src), Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = LOCAL_TEMP / f"reference_{uuid.uuid4().hex}.wav"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(src),
        "-vn", "-ac", "1", "-ar", "24000", "-c:a", "pcm_s16le", str(temp)], check=True)
    try:
        duration = _audio_duration(temp)
        if not (3.0 <= duration <= 15.0):
            raise ValueError("Reference should be 3–15 seconds, ideally 4–8 seconds of clean speech.")
        _atomic_copy(temp, dst)
        return duration
    finally:
        temp.unlink(missing_ok=True)


def _voice_folder(name):
    return VOICE_DIR / _safe_name(name)


def _voice_paths(name):
    folder = _voice_folder(name)
    return {"folder":folder, "meta":folder / "metadata.json", "ref":folder / "reference.wav"}


def list_voices():
    if not VOICE_DIR.exists():
        return []
    return sorted((f.name for f in VOICE_DIR.iterdir() if f.is_dir() and
        (f / "metadata.json").is_file() and (f / "reference.wav").is_file()), key=str.lower)


def _voice_fingerprint(meta, paths):
    h = hashlib.sha256()
    h.update(_sha256_file(paths["ref"]).encode())
    h.update(_clean_text(meta["reference_transcript"]).encode("utf-8"))
    h.update(MODEL_REPO.encode())
    h.update(meta.get("voice_mode", DEFAULT_VOICE_MODE).encode())
    return h.hexdigest()


def create_voice_profile(name, ref_audio, ref_text, language="en", default_speed=1.0, voice_mode=DEFAULT_VOICE_MODE):
    global ACTIVE_VOICE
    name = _safe_name(name)
    lang = language_name(language)
    transcript = _clean_text(ref_text)
    voice_mode = _validate_mode(voice_mode)
    if not transcript:
        raise ValueError("Enter the exact transcript of your reference clip for faithful Qwen3 ICL voice cloning.")
    src = _coerce_path(ref_audio)
    if src is None or not src.is_file():
        raise FileNotFoundError("Reference audio not found.")
    speed = float(default_speed)
    if not 0.75 <= speed <= 1.25:
        raise ValueError("Default speed must be between 0.75 and 1.25.")
    paths = _voice_paths(name)
    paths["folder"].mkdir(parents=True, exist_ok=True)
    duration = _convert_reference_audio(src, paths["ref"])
    meta = {"format":VOICE_FORMAT,"voice_name":name,"model":MODEL_REPO,
        "language":lang,"default_speed":speed,"voice_mode":voice_mode,"reference_seconds":round(duration,3),
        "reference_transcript":transcript,"reference_sha256":_sha256_file(paths["ref"]),
        "created_at":time.strftime("%Y-%m-%d %H:%M:%S")}
    _atomic_json(paths["meta"], meta)
    # Keep portable raw reference rather than serializing unsafe/internal prompt tensors.
    if MODEL is not None and hasattr(MODEL, "_voice_prompt_cache"):
        MODEL._voice_prompt_cache.clear()
        ACTIVE_VOICE = None
    return name, f"Saved '{name}' ({duration:.1f}s, {voice_mode}). Reference and transcript persist on Drive."


def load_voice_payload(name):
    paths = _voice_paths(name)
    if not paths["meta"].is_file() or not paths["ref"].is_file():
        raise FileNotFoundError(f"Saved voice {name!r} is missing metadata or reference WAV.")
    meta = json.loads(paths["meta"].read_text(encoding="utf-8"))
    if meta.get("format") != VOICE_FORMAT or not _clean_text(meta.get("reference_transcript")):
        raise ValueError(f"Voice {name!r} is not a valid Qwen3 studio profile.")
    meta["voice_mode"] = _validate_mode(meta.get("voice_mode", DEFAULT_VOICE_MODE))
    return {"reference":str(paths["ref"]), "transcript":meta["reference_transcript"], "voice_mode":meta["voice_mode"]}, meta, _voice_fingerprint(meta, paths), paths


def _activate_voice(payload):
    global ACTIVE_VOICE
    if MODEL is None:
        load_model()
    key = (payload["reference"], payload["transcript"], payload["voice_mode"])
    # Retain exactly one voice reference on GPU to avoid accidental T4 VRAM
    # exhaustion. FasterQwen3TTS memoizes the prompt on first generation.
    if ACTIVE_VOICE != key:
        if hasattr(MODEL, "_voice_prompt_cache"):
            MODEL._voice_prompt_cache.clear()
        ACTIVE_VOICE = key

def export_voice_zip(name):
    payload, meta, fp, paths = load_voice_payload(name)
    del payload
    out = EXPORT_DIR / f"{_safe_name(name)}_Qwen3.zip"
    temp = LOCAL_TEMP / f"voice_export_{uuid.uuid4().hex}.zip"
    try:
        with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED) as z:
            z.write(paths["meta"], "metadata.json")
            z.write(paths["ref"], "reference.wav")
        _atomic_copy(temp, out)
    finally:
        temp.unlink(missing_ok=True)
    return str(out)


def _validate_zip(z):
    entries = z.infolist()
    if len(entries) > MAX_IMPORT_FILES:
        raise ValueError("Too many ZIP members.")
    total = 0
    for info in entries:
        total += int(info.file_size)
        if total > MAX_IMPORT_BYTES:
            raise ValueError("ZIP uncompressed size exceeds safety limit.")
        name = info.filename.replace("\\", "/")
        parts = [p for p in name.split("/") if p not in ("", ".")]
        if name.startswith("/") or ".." in parts or (parts and ":" in parts[0]):
            raise ValueError(f"Blocked unsafe ZIP path: {name}")
        mode = (info.external_attr >> 16) & 0o170000
        if mode not in (0, 0o100000, 0o040000):
            raise ValueError(f"Blocked nonregular ZIP member: {name}")


def import_voice_zip(zip_file):
    path = _coerce_path(zip_file)
    if not path or not path.is_file() or path.suffix.lower() != ".zip":
        raise ValueError("Upload a .zip voice archive.")
    work = LOCAL_TEMP / f"import_{uuid.uuid4().hex}"
    work.mkdir(parents=True)
    installed = []
    try:
        with zipfile.ZipFile(path) as z:
            _validate_zip(z)
            z.extractall(work)
        for meta_path in work.rglob("metadata.json"):
            root = meta_path.parent
            ref = root / "reference.wav"
            if not ref.is_file():
                continue
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            transcript = _clean_text(meta.get("reference_transcript"))
            if not transcript:
                continue
            name, _ = create_voice_profile(meta.get("voice_name") or root.name,
                ref, transcript, meta.get("language") or "English", meta.get("default_speed") or 1.0, meta.get("voice_mode") or DEFAULT_VOICE_MODE)
            installed.append(name)
        if not installed:
            raise ValueError("ZIP needs metadata.json with reference_transcript and reference.wav. "
                "Old CosyVoice prompt.pt tensors alone cannot be converted to Qwen3-TTS.")
        return installed
    finally:
        shutil.rmtree(work, ignore_errors=True)


def migrate_existing_cosyvoice_profiles():
    """Import original WAV + transcript from CosyVoice3 and Ming3 folders without deleting anything."""
    imported = []
    for old_dir in [Path("/content/drive/MyDrive/CosyVoice3_Production/voices"),
                    Path("/content/drive/MyDrive/Ming3_Production/voices")]:
        if not old_dir.is_dir():
            continue
        for folder in sorted(old_dir.iterdir()):
            if not folder.is_dir() or folder.name in list_voices():
                continue
            ref, meta_path = folder / "reference.wav", folder / "metadata.json"
            if not (ref.is_file() and meta_path.is_file()):
                continue
            try:
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
                if not meta.get("reference_transcript"):
                    continue
                new, _ = create_voice_profile(folder.name, ref, meta["reference_transcript"],
                    meta.get("language", "en"), meta.get("default_speed", 1.0),
                    DEFAULT_VOICE_MODE)
                imported.append(new)
            except Exception as exc:
                print(f"Skipping legacy voice {folder.name}: {exc}", flush=True)
    return imported


# -----------------------------------------------------------------------------
# Sentence-safe splitter: copied from the proven OmniVoice workflow, with no
# character/word fallback. One overlong sentence is refused rather than cut.
# -----------------------------------------------------------------------------
SPLITTER_VERSION = "sentence_safe_v7_qwen3"
TARGET_CHARS_PER_CHUNK = 200
MAX_SENTENCE_CHARS = 300
_ABBREVIATIONS = {
    "mr", "mrs", "ms", "dr", "prof", "sr", "jr", "st", "vs", "etc", "e.g", "i.e", "no",
    "sra", "srta", "dra", "ud", "uds", "m", "mme", "mlle", "bzw", "usw", "z.b", "u.a",
}
_CLOSERS = set('"\'”’»)]}')
_END_MARKS = set(".!?…。！？")


def _previous_word(text, dot_index):
    left = text[:dot_index]
    m = re.search(r"([A-Za-zÀ-ÖØ-öø-ÿ]+(?:\.[A-Za-zÀ-ÖØ-öø-ÿ]+)*)$", left)
    return (m.group(1) if m else "").lower()


def _period_is_sentence_end(text, i):
    if i > 0 and i + 1 < len(text) and text[i - 1].isdigit() and text[i + 1].isdigit():
        return False
    token = _previous_word(text, i)
    if token in _ABBREVIATIONS:
        return False
    if token and len(token.split(".")[-1]) == 1:
        j = i + 1
        while j < len(text) and text[j].isspace():
            j += 1
        if j < len(text) and (text[j].isupper() or text[j].isdigit()):
            return False
    return True


def split_sentences(text):
    text = clean_script(text)
    if not text:
        return []
    sentences, start, i, n = [], 0, 0, len(text)
    while i < n:
        ch = text[i]
        is_end = ch in _END_MARKS
        if ch == "." and is_end:
            is_end = _period_is_sentence_end(text, i)
        if not is_end:
            i += 1
            continue
        j = i + 1
        while j < n and text[j] in _END_MARKS:
            j += 1
        while j < n and text[j] in _CLOSERS:
            j += 1
        if j == n or text[j].isspace():
            sentence = text[start:j].strip()
            if sentence:
                sentences.append(sentence)
            while j < n and text[j].isspace():
                j += 1
            start, i = j, j
        else:
            i += 1
    tail = text[start:].strip()
    if tail:
        sentences.append(tail)
    return sentences


def split_script(text, target_chars=TARGET_CHARS_PER_CHUNK, max_sentence_chars=MAX_SENTENCE_CHARS):
    sentences = split_sentences(text)
    if not sentences:
        return []
    for idx, sentence in enumerate(sentences, start=1):
        if len(sentence) > int(max_sentence_chars):
            preview = sentence[:140] + ("…" if len(sentence) > 140 else "")
            raise ValueError(
                f"Sentence {idx} is {len(sentence)} characters, above the safe {max_sentence_chars}-character limit. "
                "This notebook will not cut a sentence in the middle. Add a full stop/question mark/exclamation mark "
                f"inside that sentence and retry. Preview: {preview}"
            )
    chunks, current = [], ""
    for sentence in sentences:
        candidate = sentence if not current else f"{current} {sentence}"
        if current and len(candidate) > int(target_chars):
            chunks.append(current)
            current = sentence
        else:
            current = candidate
    if current:
        chunks.append(current)
    if clean_script(" ".join(chunks)) != clean_script(text):
        raise RuntimeError("Sentence splitter integrity check failed; generation was stopped.")
    return chunks




def _valid_wav(path, expected_sr=None):
    expected_sr = int(expected_sr or SAMPLE_RATE)
    try:
        info = sf.info(str(path))
        return info.frames > 0 and info.channels == 1 and int(info.samplerate) == expected_sr
    except Exception:
        return False


def _trim_excess_edge_silence(arr, sr):
    arr = np.asarray(arr, dtype=np.float32).reshape(-1)
    if arr.size < sr:
        return arr
    frame = max(1, int(0.02 * sr))
    threshold = 10 ** (-50 / 20)
    rms = []
    for start in range(0, len(arr), frame):
        x = arr[start:start + frame]
        rms.append(float(np.sqrt(np.mean(x * x) + 1e-12)))
    active = np.flatnonzero(np.asarray(rms) > threshold)
    if active.size == 0:
        return arr
    first, last = int(active[0]), int(active[-1])
    lead = first * frame
    tail = len(arr) - min(len(arr), (last + 1) * frame)
    keep_lead = int(0.12 * sr)
    keep_tail = int(0.18 * sr)
    start = 0 if lead <= int(0.60 * sr) else max(0, lead - keep_lead)
    end = len(arr) if tail <= int(0.75 * sr) else min(len(arr), len(arr) - tail + keep_tail)
    return arr[start:end]


def _audio_guard(arr, text, sr):
    arr = np.asarray(arr, dtype=np.float32).reshape(-1)
    if arr.size == 0 or not np.isfinite(arr).all():
        return False, "empty/non-finite audio"
    duration = len(arr) / float(sr)
    if duration < 0.35:
        return False, f"implausibly short audio ({duration:.2f}s)"
    peak = float(np.max(np.abs(arr)))
    rms = float(np.sqrt(np.mean(arr * arr) + 1e-12))
    if rms < 0.0015:
        return False, "near-silent output"
    clipping = float(np.mean(np.abs(arr) >= 0.999))
    if clipping > 0.01:
        return False, f"heavy clipping ({clipping*100:.2f}%)"

    chars = len(re.sub(r"\s+", "", text))
    if chars >= 80:
        cps = chars / max(duration, 1e-6)
        if cps < 3.0:
            return False, f"audio unusually long for text ({cps:.1f} chars/s)"
        if cps > 35.0:
            return False, f"audio unusually short for text ({cps:.1f} chars/s)"

    # Detect pathological long dropouts, not normal sentence pauses.
    frame = max(1, int(0.05 * sr))
    threshold = 10 ** (-52 / 20)
    longest = run = 0
    for start in range(0, len(arr), frame):
        x = arr[start:start + frame]
        frame_rms = float(np.sqrt(np.mean(x * x) + 1e-12))
        if frame_rms < threshold:
            run += 1
            longest = max(longest, run)
        else:
            run = 0
    longest_silence = longest * frame / float(sr)
    if chars >= 40 and longest_silence > 3.0:
        return False, f"long internal silence ({longest_silence:.1f}s)"
    return True, "ok"


def _write_checkpoint_wav(destination, audio):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    local = LOCAL_TEMP / f"chunk_{uuid.uuid4().hex}.wav"
    drive_tmp = destination.with_name(f".{destination.name}.{uuid.uuid4().hex}.tmp")
    try:
        arr = _trim_excess_edge_silence(audio, SAMPLE_RATE)
        if arr.size == 0 or not np.isfinite(arr).all():
            raise RuntimeError("Model returned invalid audio samples.")
        peak = float(np.max(np.abs(arr)))
        if peak > 0.995:
            arr = arr * (0.985 / peak)
        sf.write(str(local), arr, SAMPLE_RATE, format="WAV", subtype="PCM_16")
        if not _valid_wav(local):
            raise RuntimeError("Local WAV validation failed.")
        shutil.copy2(local, drive_tmp)
        if not _valid_wav(drive_tmp):
            raise RuntimeError("Drive WAV checkpoint validation failed.")
        os.replace(drive_tmp, destination)
    finally:
        local.unlink(missing_ok=True)
        drive_tmp.unlink(missing_ok=True)





def _atempo_pitch_preserving(arr, speed):
    """Qwen3 base clone uses external pitch-preserving FFmpeg atempo."""
    speed = float(speed)
    if abs(speed-1.0) < 0.0001:
        return arr
    if not 0.75 <= speed <= 1.25:
        raise ValueError("Speed must be between 0.75 and 1.25")
    work = LOCAL_TEMP / f"tempo_{uuid.uuid4().hex}"
    work.mkdir()
    try:
        inp, out = work / "input.wav", work / "output.wav"
        sf.write(str(inp), arr, SAMPLE_RATE, subtype="FLOAT")
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(inp),
            "-filter:a", f"atempo={speed:.4f}", "-ar", str(SAMPLE_RATE),
            "-ac", "1", "-c:a", "pcm_f32le", str(out)], check=True)
        changed, sr = sf.read(str(out), dtype="float32")
        if sr != SAMPLE_RATE or changed.size == 0:
            raise RuntimeError("Speed adjustment produced empty/unexpected audio.")
        return changed
    finally:
        shutil.rmtree(work, ignore_errors=True)


def _light_normalize_text(text, language):
    # Text normalization is an optional deterministic aid, not a separate LLM.
    import unicodedata
    text = unicodedata.normalize("NFKC", clean_script(text))
    text = text.replace(" ", " ").replace("‘", "'").replace("’", "'")
    # Numbers may denote software versions / dates: do not automatically
    # verbalize them, since the intended reading is context-dependent.
    return text


def _generate_chunk_once(text, ref_path, transcript, language, speed, quality,
                         normalize_text, seed, voice_mode=DEFAULT_VOICE_MODE):
    if MODEL is None:
        load_model()
    language = language_name(language)
    mode = _validate_mode(voice_mode)
    random.seed(seed)
    np.random.seed(seed & 0xFFFFFFFF)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    content = _light_normalize_text(text, language) if normalize_text else clean_script(text)
    with MODEL_LOCK, torch.inference_mode():
        wavs, sr = MODEL.generate_voice_clone(
            text=content, language=language, ref_audio=str(ref_path),
            ref_text=str(transcript), xvec_only=(mode == VOICE_MODES[1]),
            append_silence=True, max_new_tokens=512,
            temperature=0.9, top_k=50, top_p=1.0,
            repetition_penalty=1.05, do_sample=True,
        )
    if not wavs:
        raise RuntimeError("Qwen3-TTS returned no waveform")
    if int(sr) != SAMPLE_RATE:
        raise RuntimeError(f"Qwen3-TTS returned sample rate {sr} instead of {SAMPLE_RATE}")
    arr = np.asarray(wavs[0], dtype=np.float32).reshape(-1)
    if arr.size == 0 or not np.isfinite(arr).all():
        raise RuntimeError("Qwen3-TTS returned empty or non-finite audio.")
    return _atempo_pitch_preserving(arr, speed)

def _chunk_seed(job_id, chunk_index, attempt=0):
    return int(hashlib.sha256(f"{job_id}:{chunk_index}:{attempt}".encode()).hexdigest()[:8], 16)


def _generate_chunk_guarded(text, ref_path, transcript, language, speed, quality,
                            normalize_text, job_id, chunk_index, voice_mode=DEFAULT_VOICE_MODE):
    retries = QUALITY_PRESETS.get(quality, QUALITY_PRESETS[DEFAULT_QUALITY])["retries"]
    last_reason, last_exc = None, None
    for attempt in range(retries + 1):
        try:
            audio = _generate_chunk_once(text, ref_path, transcript, language, speed,
                quality, normalize_text, _chunk_seed(job_id, chunk_index, attempt), voice_mode)
            ok, reason = _audio_guard(audio, text, SAMPLE_RATE)
            if ok:
                return audio, attempt
            last_reason = reason
            print(f"⚠️ Chunk {chunk_index + 1} failed guard: {reason} (attempt {attempt+1})", flush=True)
        except Exception as exc:
            last_exc = exc
            if _is_fatal_cuda_error(exc):
                raise RuntimeError(_cuda_launch_guidance(exc)) from exc
            print(f"⚠️ Chunk {chunk_index + 1} attempt {attempt+1}: {type(exc).__name__}: {exc}", flush=True)
        _release_memory()
    if last_exc:
        raise RuntimeError(f"Chunk failed after {retries+1} attempt(s): {last_exc}") from last_exc
    raise RuntimeError(f"Chunk failed audio guard after {retries+1} attempt(s): {last_reason}")


def _natural_key(value):
    name = _coerce_path(value).name.lower()
    return [int(x) if x.isdigit() else x for x in re.split(r"(\d+)", name)]


def _read_txt(path):
    path = _coerce_path(path)
    if not path or not path.exists():
        raise FileNotFoundError("TXT file is no longer available.")
    last = None
    for enc in ("utf-8-sig", "utf-8", "cp1252"):
        try:
            return path.read_text(encoding=enc)
        except UnicodeDecodeError as exc:
            last = exc
    raise ValueError(f"Could not decode {path.name}: {last}")


def _job_identity(text, voice_fp, language, speed, quality, normalize_text, mastering):
    preset = QUALITY_PRESETS.get(quality, QUALITY_PRESETS[DEFAULT_QUALITY])
    payload = "\n".join([
        CACHE_VERSION, MODEL_REPO, MODEL_SNAPSHOT_ID, SPLITTER_VERSION, QUALITY_GUARD_VERSION,
        voice_fp, str(language), f"{float(speed):.4f}", str(quality),
        f"Qwen3 fp32 sdpa {INFERENCE_BACKEND} temp=0.9 topk=50 rep=1.05",
        str(bool(normalize_text)), str(bool(mastering)), clean_script(text),
    ])
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def _job_dir(text, voice_fp, language, speed, quality, normalize_text, mastering, source_name):
    jid = _job_identity(text, voice_fp, language, speed, quality, normalize_text, mastering)
    return CACHE_DIR / f"{_safe_stem(source_name)}_{jid[:16]}", jid


def _status(job_dir, **payload):
    _atomic_json(Path(job_dir) / "status.json", {"updated": time.time(), **payload})


def _make_silence(path, ms):
    frames = max(1, int(SAMPLE_RATE * float(ms) / 1000.0))
    sf.write(str(path), np.zeros(frames, dtype=np.float32), SAMPLE_RATE, subtype="PCM_16")


def _merge_chunks_to_mp3(chunk_paths, output_path, pause_ms=DEFAULT_PAUSE_MS, mastering=True):
    chunk_paths = [Path(p) for p in chunk_paths]
    if not chunk_paths:
        raise ValueError("No chunks to merge.")
    for p in chunk_paths:
        if not _valid_wav(p):
            raise RuntimeError(f"Invalid WAV checkpoint: {p.name}")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    work = LOCAL_TEMP / f"merge_{uuid.uuid4().hex}"
    work.mkdir(parents=True, exist_ok=False)
    local_mp3 = work / "final.mp3"
    concat_file = work / "concat.txt"
    silence = work / "silence.wav"
    try:
        _make_silence(silence, pause_ms)
        with concat_file.open("w", encoding="utf-8") as f:
            for idx, wav in enumerate(chunk_paths):
                escaped = str(wav.resolve()).replace("'", "'\\''")
                f.write(f"file '{escaped}'\n")
                if pause_ms > 0 and idx != len(chunk_paths) - 1:
                    s = str(silence.resolve()).replace("'", "'\\''")
                    f.write(f"file '{s}'\n")
        cmd = [
            "ffmpeg", "-y", "-loglevel", "error", "-f", "concat", "-safe", "0",
            "-i", str(concat_file), "-vn", "-ar", str(SAMPLE_RATE), "-ac", "1",
        ]
        if mastering:
            cmd += ["-af", "highpass=f=55,loudnorm=I=-16:LRA=9:TP=-1.5"]
        cmd += ["-c:a", "libmp3lame", "-b:a", MP3_BITRATE, str(local_mp3)]
        subprocess.run(cmd, check=True)
        if not local_mp3.exists() or local_mp3.stat().st_size < 1000:
            raise RuntimeError("FFmpeg produced an empty MP3.")
        _atomic_copy(local_mp3, output_path)
    finally:
        shutil.rmtree(work, ignore_errors=True)


def _mp3_duration(path):
    cmd = [
        "ffprobe", "-v", "error", "-show_entries", "format=duration",
        "-of", "default=noprint_wrappers=1:nokey=1", str(path),
    ]
    p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if p.returncode != 0:
        raise RuntimeError(f"ffprobe failed for {Path(path).name}: {p.stderr[-500:]}")
    return float(p.stdout.strip())


def _process_script(
    text, source_name, voice_name, payload, meta, voice_fp,
    quality=DEFAULT_QUALITY, speed=None, normalize_text=False,
    mastering=True, progress=None, file_index=0, total_files=1,
    language_override="Voice default",
):
    text = clean_script(text)
    if not text:
        raise ValueError(f"{source_name}: script is empty.")
    chunks = split_script(text)
    if not chunks:
        raise ValueError(f"{source_name}: no sentence chunks were created.")
    language = resolve_language(meta, language_override)
    speed = float(meta.get("default_speed", 1.0) if speed is None else speed)
    job_dir, jid = _job_dir(text, voice_fp, language, speed, quality, normalize_text, mastering, source_name)
    chunks_dir = job_dir / "chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)
    output = OUTPUT_DIR / f"{_safe_stem(source_name)}__{jid[:8]}.mp3"
    done = job_dir / "done.json"

    if done.exists() and output.exists() and output.stat().st_size > 1000:
        return {
            "source": source_name, "output": str(output), "chunks": len(chunks),
            "reused": True, "audio_seconds": _mp3_duration(output), "job_id": jid,
        }

    _atomic_json(job_dir / "job.json", {
        "cache_version": CACHE_VERSION,
        "job_id": jid,
        "source": source_name,
        "voice": voice_name,
        "voice_fingerprint": voice_fp,
        "language": language,
        "quality": quality,
        "retries": QUALITY_PRESETS.get(quality, QUALITY_PRESETS[DEFAULT_QUALITY])["retries"],
        "speed": speed,
        "normalize_text": bool(normalize_text),
        "mastering": bool(mastering),
        "splitter": SPLITTER_VERSION,
        "quality_guard": QUALITY_GUARD_VERSION,
        "chunk_count": len(chunks),
        "chunk_lengths": [len(c) for c in chunks],
        "chunk_sha256": [hashlib.sha256(c.encode("utf-8")).hexdigest() for c in chunks],
    })

    _activate_voice(payload)
    paths = _voice_paths(voice_name)
    chunk_paths = [chunks_dir / f"{i:04d}.wav" for i in range(len(chunks))]
    for p in chunk_paths:
        if p.exists() and not _valid_wav(p):
            p.unlink(missing_ok=True)

    started = time.time()
    for idx, (chunk, chunk_path) in enumerate(zip(chunks, chunk_paths)):
        if _valid_wav(chunk_path):
            if progress is not None:
                global_frac = (file_index + (idx + 1) / len(chunks)) / max(1, total_files)
                progress(global_frac, desc=f"[{file_index+1}/{total_files}] {source_name} — chunk {idx+1}/{len(chunks)} cached")
            continue

        if progress is not None:
            global_frac = (file_index + idx / len(chunks)) / max(1, total_files)
            progress(global_frac, desc=f"[{file_index+1}/{total_files}] {source_name} — generating chunk {idx+1}/{len(chunks)}")
        _status(
            job_dir, state="generating", file_index=file_index + 1, total_files=total_files,
            chunk_index=idx + 1, total_chunks=len(chunks),
        )
        audio, retries_used = _generate_chunk_guarded(
            text=chunk, ref_path=paths["ref"], transcript=payload["transcript"], language=language, speed=speed, quality=quality,
            normalize_text=normalize_text, job_id=jid, chunk_index=idx, voice_mode=payload["voice_mode"],
        )
        _write_checkpoint_wav(chunk_path, audio)
        del audio
        _release_memory(force=False)

    if progress is not None:
        progress((file_index + 0.97) / max(1, total_files), desc=f"[{file_index+1}/{total_files}] {source_name} — assembling MP3")
    _merge_chunks_to_mp3(chunk_paths, output, mastering=mastering)
    audio_seconds = _mp3_duration(output)
    report = {
        "source": source_name,
        "output": str(output),
        "chunks": len(chunks),
        "reused": False,
        "audio_seconds": audio_seconds,
        "wall_seconds": time.time() - started,
        "rtf": (time.time() - started) / max(audio_seconds, 0.001),
        "job_id": jid,
    }
    _atomic_json(done, report)
    _status(job_dir, state="complete", **report)
    return report


def generate_single(text, txt_file, voice_name, quality, speed, normalize_text, mastering, progress=None, language_override="Voice default"):
    if txt_file:
        path = _coerce_path(txt_file)
        text = _read_txt(path)
        source = path.name
    else:
        source = "single_voiceover.txt"
    if not voice_name:
        raise ValueError("Select a cloned voice.")
    with MODEL_LOCK:
        payload, meta, voice_fp, paths = load_voice_payload(voice_name)
        return _process_script(
            text=text, source_name=source, voice_name=voice_name,
            payload=payload, meta=meta, voice_fp=voice_fp,
            quality=quality, speed=speed, normalize_text=normalize_text,
            mastering=mastering, progress=progress, file_index=0, total_files=1,
            language_override=language_override,
        )


def generate_batch_files(txt_files, voice_name, quality, speed, normalize_text, mastering, progress=None, language_override="Voice default"):
    if not txt_files:
        raise ValueError("Upload one or more TXT files.")
    if not voice_name:
        raise ValueError("Select a cloned voice.")
    paths = sorted([_coerce_path(x) for x in txt_files], key=_natural_key)
    results, errors = [], []
    with MODEL_LOCK:
        payload, meta, voice_fp, voice_paths = load_voice_payload(voice_name)
        _activate_voice(payload)
        for file_index, path in enumerate(paths):
            try:
                report = _process_script(
                    text=_read_txt(path), source_name=path.name, voice_name=voice_name,
                    payload=payload, meta=meta, voice_fp=voice_fp,
                    quality=quality, speed=speed, normalize_text=normalize_text,
                    mastering=mastering, progress=progress,
                    file_index=file_index, total_files=len(paths),
                    language_override=language_override,
                )
                results.append(report)
            except Exception as exc:
                if _is_fatal_cuda_error(exc):
                    raise RuntimeError(
                        f"Fatal CUDA state while processing {path.name}: {exc}. Restart the runtime, rerun the notebook, "
                        "then upload the same TXT files; completed checkpoints will resume."
                    ) from exc
                errors.append({"source": path.name, "error": f"{type(exc).__name__}: {exc}"})
                _release_memory(force=True)
                continue
    if progress is not None:
        progress(1.0, desc="Batch finished")
    return results, errors





def run_qa():
    """Deterministic tests that don't require downloading model weights or a GPU."""
    import io
    checks=[]
    def check(name, fn):
        try:
            fn()
            checks.append((name, True))
            print("✅", name)
        except Exception as exc:
            checks.append((name, False))
            print("❌", name, "—", type(exc).__name__, str(exc))
    def splitter():
        txt="Dr. Rivera arrived at 8.45 p.m. Then he left! Next topic?"
        parts=split_script(txt, target_chars=36)
        assert clean_script(" ".join(parts)) == clean_script(txt)
    def too_long():
        try:
            split_script("A" * (MAX_SENTENCE_CHARS+10) + ".")
        except ValueError:
            return
        raise AssertionError("Expected sentence length validation")
    def temp_file():
        p=LOCAL_TEMP / f"qacheck_{uuid.uuid4().hex}.json"
        try:
            _atomic_json(p,{"safe":True})
            assert json.loads(p.read_text())["safe"]
        finally:
            p.unlink(missing_ok=True)
    def wav_check():
        p=LOCAL_TEMP / f"qacheck_{uuid.uuid4().hex}.wav"
        try:
            sf.write(str(p), np.zeros(SAMPLE_RATE//2), SAMPLE_RATE, subtype="PCM_16")
            assert _valid_wav(p)
        finally:
            p.unlink(missing_ok=True)
    def zip_guard():
        p=LOCAL_TEMP / f"qacheck_{uuid.uuid4().hex}.zip"
        try:
            with zipfile.ZipFile(p,"w") as z:
                z.writestr("../evil.txt", "blocked")
            with zipfile.ZipFile(p) as z:
                try:
                    _validate_zip(z)
                except ValueError:
                    return
            raise AssertionError("ZIP traversal not rejected")
        finally:
            p.unlink(missing_ok=True)
    def seeded():
        assert _chunk_seed("test", 2, 0) != _chunk_seed("test", 2, 1)
    def language():
        assert resolve_language({"language":"en"},"Spanish") == "Spanish"
        assert resolve_language({"language":"French"},"Voice default") == "French"
    def format_guard():
        assert VOICE_FORMAT.startswith("qwen3_") and "CosyVoice" not in MODEL_REPO
    def tempo():
        assert np.array_equal(_atempo_pitch_preserving(np.array([0,1],dtype="float32"),1.0),[0,1])
    for name, func in [
        ("sentence integrity",splitter),("oversize sentence rejected",too_long),
        ("atomic JSON",temp_file),("wav checkpoint",wav_check),
        ("ZIP traversal rejection",zip_guard),("unique retry seeds",seeded),
        ("language routing",language),("Qwen storage format",format_guard),
        ("speed 1.0 transparent",tempo),
    ]:
        check(name,func)
    if not all(ok for _, ok in checks):
        raise RuntimeError("Studio QA failed")
    return checks


def launch_ui():
    if gr is None:
        raise RuntimeError("Gradio is not installed.")

    # Build a fresh Gradio app for this process.
    # No previously-created Blocks object needs to be closed here.
    def _voice_choices():
        return list_voices()

    def _voice_update(current=None):
        choices = _voice_choices()
        value = current if current in choices else (choices[0] if choices else None)
        return gr.update(choices=choices, value=value)

    def ui_refresh(current_manager=None, current_single=None, current_batch=None):
        voices = _voice_choices()
        status = "Available voices: " + (", ".join(voices) if voices else "none yet")
        return status, _voice_update(current_manager), _voice_update(current_single), _voice_update(current_batch)

    def ui_live_dropdowns(v, s, b):
        return _voice_update(v), _voice_update(s), _voice_update(b)

    def ui_transcribe_voice(ref_file, language):
        try:
            text = transcribe_reference_audio(ref_file, language)
            return text, "✅ Auto-transcript created. Please correct it so it exactly matches the spoken reference."
        except Exception as exc:
            return gr.update(), f"❌ {type(exc).__name__}: {exc}"

    def ui_save_voice(name, language, transcript, ref_file, default_speed, voice_mode):
        try:
            with MODEL_LOCK, VOICE_IO_LOCK:
                saved, detail = create_voice_profile(name, ref_file, transcript, language, default_speed, voice_mode)
            voices = _voice_choices()
            status = f"✅ {detail}\nVoices: {', '.join(voices)}"
            return status, _voice_update(saved), _voice_update(saved), _voice_update(saved)
        except Exception as exc:
            return f"❌ {type(exc).__name__}: {exc}", gr.update(), gr.update(), gr.update()

    def ui_export_voice(name):
        try:
            if not name:
                raise ValueError("Select a voice first.")
            with VOICE_IO_LOCK:
                path = export_voice_zip(name)
            return f"✅ Exported {name}", path
        except Exception as exc:
            return f"❌ {type(exc).__name__}: {exc}", None

    def ui_import_voice(zip_file):
        try:
            with MODEL_LOCK, VOICE_IO_LOCK:
                names = import_voice_zip(zip_file)
            chosen = names[0] if names else None
            return f"✅ Imported: {', '.join(names)}", _voice_update(chosen), _voice_update(chosen), _voice_update(chosen)
        except Exception as exc:
            return f"❌ {type(exc).__name__}: {exc}", gr.update(), gr.update(), gr.update()

    def ui_single(text, txt_file, voice, language, quality, speed, normalize_text, mastering, progress=gr.Progress()):
        try:
            report = generate_single(
                text=text, txt_file=txt_file, voice_name=voice, quality=quality,
                speed=float(speed), normalize_text=normalize_text,
                mastering=mastering, progress=progress, language_override=language,
            )
            reused = " (reused completed cache)" if report.get("reused") else ""
            rtf = report.get("rtf")
            rtf_line = f"\nRTF: {rtf:.2f}" if isinstance(rtf, (int, float)) else ""
            status = (
                f"✅ Complete{reused}\n"
                f"Chunks: {report['chunks']}\n"
                f"Audio: {report['audio_seconds']/60:.2f} min{rtf_line}\n"
                f"Output: {report['output']}"
            )
            return status, report["output"]
        except Exception as exc:
            return f"❌ {type(exc).__name__}: {exc}", None

    def ui_batch(files, voice, language, quality, speed, normalize_text, mastering, _worker, progress=gr.Progress()):
        try:
            results, errors = generate_batch_files(
                txt_files=files, voice_name=voice, quality=quality,
                speed=float(speed), normalize_text=normalize_text,
                mastering=mastering, progress=progress, language_override=language,
            )
            lines = [f"✅ Completed: {len(results)} | Failed: {len(errors)}"]
            for r in results:
                tag = "reused" if r.get("reused") else "generated"
                rtf = f" | RTF {r['rtf']:.2f}" if isinstance(r.get("rtf"), (int, float)) else ""
                lines.append(f"✅ {r['source']} → {Path(r['output']).name} | {r['chunks']} chunks | {tag}{rtf}")
            for e in errors:
                lines.append(f"❌ {e['source']} — {e['error']}")
            return "\n".join(lines), [r["output"] for r in results]
        except Exception as exc:
            return f"❌ {type(exc).__name__}: {exc}", []

    voices_initial = _voice_choices()
    voice_default = voices_initial[0] if voices_initial else None

    with gr.Blocks(title="Qwen3-TTS Professional Clone Studio — T4") as demo:
        gr.Markdown("# 🎙️ Qwen3-TTS Professional Clone Studio — T4 FP32 Edition")
        gr.Markdown(
            f"Qwen3-TTS 0.6B on Tesla T4 (FP32 + SDPA, {INFERENCE_BACKEND} backend). "
            "Voice cloning (ICL / x-vector), single or batch TXT, sentence-safe chunks, retries, Drive checkpoints and MP3 exports."
        )

        with gr.Tabs():
            with gr.Tab("Voice Manager"):
                with gr.Row():
                    with gr.Column(scale=2):
                        vm_name = gr.Textbox(label="Voice name", placeholder="Black Fatigue")
                        vm_lang = gr.Textbox(label="Language code/name", value="en", info="Voice default language; can override on generation tabs")
                        vm_speed = gr.Slider(0.75, 1.25, value=1.0, step=0.01, label="Default speaking speed")
                        vm_mode = gr.Dropdown(choices=VOICE_MODES, value=DEFAULT_VOICE_MODE, label="Voice cloning mode", info="ICL = highest voice fidelity; x-vector = faster multilingual switching")
                        vm_ref = gr.File(label="Reference audio (4–8 sec ideal; 3–15 sec supported)", type="filepath")
                        vm_text = gr.Textbox(label="Exact reference transcript", lines=5)
                        vm_transcribe = gr.Button("Auto-transcribe Reference (optional)")
                        vm_save = gr.Button("Create / Replace Voice", variant="primary")
                    with gr.Column(scale=2):
                        vm_select = gr.Dropdown(choices=voices_initial, value=voice_default, label="Saved voices")
                        vm_refresh = gr.Button("Refresh Voices")
                        vm_export = gr.Button("Export Selected Voice ZIP")
                        vm_export_file = gr.File(label="Exported voice ZIP")
                        vm_import_file = gr.File(label="Import Voice ZIP", type="filepath", file_types=[".zip"])
                        vm_import = gr.Button("Import Voice ZIP")
                        vm_status = gr.Textbox(label="Voice Manager status", lines=8)

            with gr.Tab("Single Generate"):
                sg_voice = gr.Dropdown(choices=voices_initial, value=voice_default, label="Voice")
                sg_language = gr.Dropdown(choices=LANGUAGES, value="Voice default", label="Generation language")
                sg_text = gr.Textbox(label="Script text", lines=12)
                sg_file = gr.File(label="Or upload one TXT", type="filepath", file_types=[".txt"])
                with gr.Row():
                    sg_quality = gr.Dropdown(list(QUALITY_PRESETS.keys()), value=DEFAULT_QUALITY, label="Reliability / retries")
                    sg_speed = gr.Slider(0.75, 1.25, value=1.0, step=0.01, label="Speaking speed")
                with gr.Row():
                    sg_norm = gr.Checkbox(value=False, label="Light Unicode/text cleanup (no number rewriting)")
                    sg_master = gr.Checkbox(value=True, label="Transparent final mastering")
                sg_go = gr.Button("Generate / Resume MP3", variant="primary")
                sg_status = gr.Textbox(label="Status", lines=8)
                sg_output = gr.File(label="Finished MP3")

            with gr.Tab("Batch Generate"):
                bg_voice = gr.Dropdown(choices=voices_initial, value=voice_default, label="Voice")
                bg_language = gr.Dropdown(choices=LANGUAGES, value="Voice default", label="Generation language")
                bg_files = gr.File(
                    label="Upload TXT files (1, 2, 3, or more)", type="filepath",
                    file_count="multiple", file_types=[".txt"],
                )
                with gr.Row():
                    bg_quality = gr.Dropdown(list(QUALITY_PRESETS.keys()), value=DEFAULT_QUALITY, label="Reliability / retries")
                    bg_speed = gr.Slider(0.75, 1.25, value=1.0, step=0.01, label="Speaking speed")
                    bg_worker = gr.Dropdown([1], value=1, label="GPU worker", interactive=False, info="Locked to 1 to prevent model-state races")
                with gr.Row():
                    bg_norm = gr.Checkbox(value=False, label="Light Unicode/text cleanup (no number rewriting)")
                    bg_master = gr.Checkbox(value=True, label="Transparent final mastering")
                bg_go = gr.Button("Generate / Resume All MP3s", variant="primary")
                bg_status = gr.Textbox(label="Batch status", lines=12)
                bg_outputs = gr.File(label="Finished MP3s", file_count="multiple")

        vm_transcribe.click(
            ui_transcribe_voice, inputs=[vm_ref, vm_lang], outputs=[vm_text, vm_status],
        )
        vm_save.click(
            ui_save_voice, inputs=[vm_name, vm_lang, vm_text, vm_ref, vm_speed, vm_mode],
            outputs=[vm_status, vm_select, sg_voice, bg_voice],
            concurrency_id="qwen3_gpu", concurrency_limit=1,
        )
        vm_export.click(
            ui_export_voice, inputs=[vm_select], outputs=[vm_status, vm_export_file],
            concurrency_id="qwen3_gpu", concurrency_limit=1,
        )
        vm_import.click(
            ui_import_voice, inputs=[vm_import_file], outputs=[vm_status, vm_select, sg_voice, bg_voice],
            concurrency_id="qwen3_gpu", concurrency_limit=1,
        )
        # Live refresh every 5 seconds: discovers voices added from another Colab tab
        # or mounted Drive without overwriting the selected voice.
        demo.load(ui_live_dropdowns, inputs=[vm_select, sg_voice, bg_voice],
                  outputs=[vm_select, sg_voice, bg_voice])
        gr.Timer(5).tick(ui_live_dropdowns, inputs=[vm_select, sg_voice, bg_voice],
                         outputs=[vm_select, sg_voice, bg_voice], show_progress="hidden")
        vm_refresh.click(
            ui_refresh, inputs=[vm_select, sg_voice, bg_voice], outputs=[vm_status, vm_select, sg_voice, bg_voice],
        )
        sg_go.click(
            ui_single, inputs=[sg_text, sg_file, sg_voice, sg_language, sg_quality, sg_speed, sg_norm, sg_master],
            outputs=[sg_status, sg_output], concurrency_id="qwen3_gpu", concurrency_limit=1,
        )
        bg_go.click(
            ui_batch, inputs=[bg_files, bg_voice, bg_language, bg_quality, bg_speed, bg_norm, bg_master, bg_worker],
            outputs=[bg_status, bg_outputs], concurrency_id="qwen3_gpu", concurrency_limit=1,
        )

    demo.queue(default_concurrency_limit=1, max_size=20)
    is_colab = importlib.util.find_spec("google.colab") is not None or Path("/content").exists()
    print("✅ Gradio app built. Launching share link…", flush=True)
    demo.launch(
        share=is_colab,
        debug=False,
        show_error=True,
        allowed_paths=[str(BASE)],
        prevent_thread_lock=False,
    )





if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--qa-only", action="store_true")
    parser.add_argument("--smoke-test", action="store_true")
    parser.add_argument("--migrate-legacy", action="store_true")
    args = parser.parse_args()
    run_qa()
    if args.qa_only:
        raise SystemExit(0)
    if args.migrate_legacy:
        print("Legacy migrated:", migrate_existing_cosyvoice_profiles())
        raise SystemExit(0)
    if args.smoke_test:
        load_model()
        print("Model load smoke test passed")
        raise SystemExit(0)
    load_model()
    print("✅ Qwen3-TTS loaded on GPU. Single-worker queue; outputs on Drive.", flush=True)
    launch_ui()


In [ ]:
# STEP 4 — Local deterministic preflight; no GPU model download required
import subprocess, os, sys
APP='/content/qwen3_t4_studio_app.py'
subprocess.run([sys.executable,APP,'--qa-only'],check=True,env=os.environ.copy())
print('✅ Preflight passed. You may optionally migrate old saved voices in the next cell.')


In [ ]:
# OPTIONAL — Bring over old CosyVoice3 / Ming voices (no original files are changed)
# Only reference.wav + accurate transcript can migrate; old .pt embeddings are incompatible.
# Uncomment this line and run ONCE if you need saved voices from previous notebooks:
# subprocess.run([sys.executable,APP,'--migrate-legacy'],check=True,env=os.environ.copy())
print('Legacy import not requested; skipped. You can also import individual saved ZIPs from Gradio.')


In [ ]:
# STEP 5 — Launch Gradio with FP32 numerically stable Qwen Base inference.
# If FP32 CUDA graphs report GPU out of memory, restart Colab and select
# 'official' instead of 'graphs'. The official backend uses no CUDA graphs
# and should use less peak VRAM, but can be significantly slower.
# After a device-side assert ALWAYS restart the Colab session first.
BACKEND = 'graphs'  # set to 'official' only if FP32 graph memory/capture fails
assert BACKEND in ('graphs','official')
import subprocess, os, sys
launch_env = os.environ.copy()
launch_env['QWEN_T4_BACKEND'] = BACKEND
proc = subprocess.Popen([sys.executable,'-u','/content/qwen3_t4_studio_app.py'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,bufsize=1,text=True,
    env=launch_env)
try:
    for line in proc.stdout:
        print(line, end='', flush=True)
except KeyboardInterrupt:
    print('Stopping studio process...')
    proc.terminate()
    raise
finally:
    if proc.poll() is not None:
        print('Studio process exit code:',proc.returncode)


## Important usage notes

1. Under **Voice Manager**, upload a clean **3–15-second** authorized reference recording, enter the **exact spoken transcript** (or run the optional Whisper transcription and correct errors), choose default language and voice mode, then click **Create / Replace Voice**.
2. For closest resemblance, use **High fidelity (ICL)** and keep the same reference for all chunks. For cross-language voiceovers where reference accent leaks, try **Fast multilingual (x-vector)** instead. These are alternative clone modes, not adjustable degrees of accuracy.
3. **Single Generate**: paste text or upload a TXT (uploaded TXT takes precedence). **Batch Generate**: upload many TXT files; files are processed serially to fit the T4, with one MP3 each.
4. Chunking is sentence-safe and limited to ~200 characters per chunk; an individual sentence longer than 300 characters requires a punctuation edit rather than slicing mid-sentence. Valid WAV chunks are stored in Drive and used to resume an interrupted job.
5. The speed slider applies *pitch-preserving postprocessing* with FFmpeg (`atempo`); it is not a native Qwen inference speed control. Quality presets control retries; they do not change model weights.
6. Data exports store **`reference.wav` + `metadata.json`**, not serialized tensors. For previously saved CosyVoice voices, the `prompt.pt` alone cannot transfer.
7. Text cleanup is conservative (Unicode and whitespace) and **does not** rewrite model names or numbers (e.g., Qwen3 0.6B), which matters for technical voiceovers.
8. **Failure recovery:** if Colab disconnects, reconnect T4, rerun install + mount + app + launch, then resubmit the identical script with the same voice/settings. The notebook reuses successfully generated chunk WAVs.
9. If Gradio says *CUDA out of memory*, restart runtime, reduce workloads to shorter sentences and keep just one browser job at a time. Do not run multiple Colab app processes on the same GPU.
10. First model start downloads ~model-weight assets and performs a CUDA graph warmup. The published T4 throughput figure is external and not guaranteed for each language/voice/segment.

**License:** Qwen3-TTS model weights are released under Apache-2.0; the faster backend is MIT. Honor recording owners' permission and platform policies when cloning voices.


### Crash repair included in v2

- `MimiConfig.rope_theta` is a **Transformers/Qwen codec compatibility** crash, not a T4 out-of-memory error. This notebook pins `transformers==5.15.1` and `qwen-tts-hf==0.1.1.post1`.
- `sox: not found` is addressed by installing `sox` and `libsox-fmt-all` in STEP 2.
- Hugging Face's unauthenticated request warning is not fatal; optionally provide your own `HF_TOKEN` for better download rate limits.
- If you already attempted a broken run, install the pinned versions, **restart session**, then rerun the setup/mount/app/launch cells. Drive voices and checkpoints are not deleted.
- The `torch_dtype` deprecated warning comes from dependencies; it is harmless and is not patched into third-party package code here.


### v3 generation crash remedy (T4)

- The old error `probability tensor contains either inf, nan or element < 0` can be caused by FP16 logits becoming non-finite. The main backend is now **FP32** (no BF16 required).
- FP32 CUDA Graphs use more VRAM, so if the *startup* fails with out-of-memory, change `BACKEND = 'official'` in the launch cell and restart the session; that switches to official Qwen FP32 decoding without CUDA Graphs.
- The T4 1.068× real-time benchmark used an optimized configuration and cannot be assumed for FP32; measure performance on your own speech samples.
- A CUDA device-side assert poisons the process; **restart Colab** instead of attempting an in-process retry.
- Storage remains `Qwen3_Production` on Drive; old output and checkpoints remain on Drive. The cache *key* was changed for this new numeric backend to prevent accidental mixing with FP16 generations.
